# Part 1: human groundedness ratings

Both members independently scored each of the 25 golden-set answers from the final run (Kaggle version 20, `outputs/golden_final.csv`) from 1 to 5: is every claim supported by the cited sources? This notebook only reads the ratings file and computes the summary; it needs no GPU.

In [1]:
import subprocess, sys, random, numpy as np
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "gpu": "cpu (no model is run)", "seed": SEED})

{'commit': '89e5157', 'python': '3.13.5', 'gpu': 'cpu (no model is run)', 'seed': 266}


In [2]:
import pandas as pd
h = pd.read_csv("../outputs/p1_human_ratings.csv")
g = pd.read_csv("../outputs/golden_final.csv")
assert h.gid.tolist() == g.gid.tolist() and (h.answer == g.answer).all(), "ratings must refer to the committed final answers"
assert h[["member1", "member2"]].isin([1, 2, 3, 4, 5]).all().all()
print("answers rated:", len(h))

answers rated: 25


In [3]:
summary = pd.DataFrame([{
    "mean_member1": round(h.member1.mean(), 2), "mean_member2": round(h.member2.mean(), 2),
    "groundedness_mean": round(h[["member1", "member2"]].mean().mean(), 2),
    "exact_agreement": round((h.member1 == h.member2).mean(), 2),
    "agreement_within_one": round(((h.member1 - h.member2).abs() <= 1).mean(), 2)}])
summary.to_csv("../outputs/p1_human_results.csv", index=False)
summary

,mean_member1,mean_member2,groundedness_mean,exact_agreement,agreement_within_one
0,4.16,4.44,4.3,0.48,1.0


In [4]:
h["mean"] = h[["member1", "member2"]].mean(axis=1)
print("lowest-rated answers:")
print(h.sort_values("mean")[["gid", "member1", "member2"]].head(5).to_string(index=False))
print("\nscore by correctness on the golden set:")
print(h.merge(g[["gid", "correct"]], on="gid").groupby("correct")["mean"].agg(["count", "mean"]).round(2))

lowest-rated answers:
gid  member1  member2
G19        2        1
G23        2        3
G18        2        3
G01        3        4
G17        4        4

score by correctness on the golden set:
         count  mean
correct             
False        3  3.00
True        22  4.48
